In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  STEP 0: Environment Setup                                   ║
# ║  Priority: 1) Google Drive (persistent) → 2) Colab runtime    ║
# ╚══════════════════════════════════════════════════════════════╝

import os, sys
from pathlib import Path

WORK_DIR = None  # will be set below

# ── Attempt 1: Mount Google Drive (for persistent storage) ──
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    DRIVE_MOUNTED = True
    print('[OK] Google Drive mounted.')
except Exception as e:
    DRIVE_MOUNTED = False
    print(f'[WARN] Could not mount Google Drive: {e}')
    print('       Will use Colab runtime ephemeral storage instead.')

# ── Determine working directory ──
if DRIVE_MOUNTED:
    PROJECT_ROOT = '/content/drive/MyDrive/UNITEN/research/genoderma_train'
    if not os.path.exists(PROJECT_ROOT):
        # Try fallback paths
        for alt in ['/content/drive/MyDrive/genoderma_train',
                    '/content/drive/MyDrive/Ganoderma_Research/genoderma_train']:
            if os.path.exists(alt):
                PROJECT_ROOT = alt
                break
    os.makedirs(PROJECT_ROOT, exist_ok=True)
    WORK_DIR = PROJECT_ROOT
    print(f'[INFO] Using Google Drive project root: {WORK_DIR}')
else:
    # No Drive — work in Colab runtime directly
    WORK_DIR = '/content/genoderma_train'
    os.makedirs(WORK_DIR, exist_ok=True)
    print(f'[INFO] Using Colab runtime directory: {WORK_DIR}')

%cd "{WORK_DIR}"
print(f'[INFO] Working directory set to: {os.getcwd()}')

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  STEP 1: Create data.yaml if missing                          ║
# ╚══════════════════════════════════════════════════════════════╝

import os
from pathlib import Path

DATA_YAML_CONTENT = """\
train: ./data/train/images
val: ./data/valid/images
test: ./data/test/images

nc: 1
names: ['Ganoderma']

roboflow:
  workspace: binus-ptpn4-ramadya
  project: ganoderma-detection-oil-palm
  version: 3
  license: CC BY 4.0
  url: https://universe.roboflow.com/binus-ptpn4-ramadya/ganoderma-detection-oil-palm/dataset/3
"""

if not os.path.exists('data.yaml'):
    with open('data.yaml', 'w') as f:
        f.write(DATA_YAML_CONTENT)
    print('[OK] data.yaml created.')
else:
    print('[OK] data.yaml already exists.')

%ls
print(f'\n[OK] Ready. Working dir: {os.getcwd()}')

/content/drive/MyDrive/UNITEN/research/genoderma_train
data/        LICENSE                README.md            train.ipynb
data.yaml    project_structure.png  README.roboflow.txt  visualize.ipynb
example.png  README.dataset.txt     requirements.txt


In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  STEP 2: Dataset — download from Roboflow if needed           ║
# ╚══════════════════════════════════════════════════════════════╝

import os, shutil
from pathlib import Path

DATASET_READY = (
    Path('data/train/images').exists() and
    Path('data/valid/images').exists() and
    Path('data/test/images').exists()
)

if DATASET_READY:
    print('[OK] Dataset already exists — skipping download.')
else:
    print('[WARN] Dataset images not found in data/.')
    print('       Attempting download from Roboflow...\n')
    
    try:
        %pip install -q roboflow
        from roboflow import Roboflow
        
        rf = Roboflow()  # uses ROBOFLOW_API_KEY env var if set
        project = rf.workspace("binus-ptpn4-ramadya").project("ganoderma-detection-oil-palm")
        version = project.version(3)
        dataset = version.download("yolov8")
        
        download_path = Path(dataset.location)
        print(f'[INFO] Downloaded to: {download_path}')
        
        # Move train/valid/test folders into project data/ directory
        for split in ['train', 'valid', 'test']:
            src = download_path / split
            dst = Path('data') / split
            if src.exists():
                if dst.exists():
                    shutil.rmtree(dst)
                shutil.move(str(src), str(dst))
                print(f'[OK] Moved {split} -> data/{split}')
        
        print('\n[OK] Dataset setup complete!')
        
    except Exception as e:
        print(f'\n[ERROR] Automatic download failed: {e}')
        print('\n── Manual Download Instructions ──')
        print('1. Get a free API key from https://roboflow.com/')
        print('2. Set it:  import os; os.environ["ROBOFLOW_API_KEY"] = "YOUR_KEY"')
        print('3. Re-run this cell')
        print('')
        print('Alternatively, download manually from:')
        print('  https://universe.roboflow.com/binus-ptpn4-ramadya/ganoderma-detection-oil-palm/dataset/3/download')
        print('  (choose "YOLOv8" format, download ZIP, extract into data/ folder)')
        raise RuntimeError('Dataset not available. Follow manual instructions above.') from e


In [1]:
%pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 35.3 MB/s eta 0:00:0000:01


In [ ]:
"""
YOLOv8n Training Script — Ganoderma Detection (Oil Palm)
Dataset  : Roboflow | binus-ptpn4-ramadya/ganoderma-detection-oil-palm v3
Model    : YOLOv8n (nano)
Export   : ONNX (opset 12, dynamic batching)
"""

import os
import torch
from pathlib import Path
from ultralytics import YOLO

# ──────────────────────────────────────────────
# 0. Config
# ──────────────────────────────────────────────
DATA_YAML   = os.path.join(os.getcwd(), "data.yaml")  # absolute path to data.yaml
PROJECT_DIR = os.path.join(os.getcwd(), "runs/ganoderma")  # where results are saved
print(f"[INFO] Working dir : {os.getcwd()}")
RUN_NAME    = "yolov8n_v1"         # sub-folder name for this run
EPOCHS      = 100
IMG_SIZE    = 640
BATCH_SIZE  = 16                   # lower to 8 if GPU VRAM < 4 GB
PATIENCE    = 20                   # early-stopping patience (epochs)
WORKERS     = 4                    # dataloader workers (0 on Windows)
DEVICE      = 0 if torch.cuda.is_available() else "cpu"

# ──────────────────────────────────────────────
# 1. Load pretrained YOLOv8n
# ──────────────────────────────────────────────
model = YOLO("yolov8n.pt")         # downloads automatically on first run

print(f"[INFO] Using device : {DEVICE}")
print(f"[INFO] Data config  : {DATA_YAML}")
print(f"[INFO] Epochs       : {EPOCHS}  |  Batch : {BATCH_SIZE}  |  Img size : {IMG_SIZE}")

# ──────────────────────────────────────────────
# 2. Train
# ──────────────────────────────────────────────
results = model.train(
    data        = DATA_YAML,
    epochs      = EPOCHS,
    imgsz       = IMG_SIZE,
    batch       = BATCH_SIZE,
    patience    = PATIENCE,
    workers     = WORKERS,
    device      = DEVICE,
    project     = PROJECT_DIR,
    name        = RUN_NAME,
    pretrained  = True,
    optimizer   = "AdamW",          # AdamW works well for small datasets
    lr0         = 0.001,            # initial learning rate
    lrf         = 0.01,             # final LR = lr0 * lrf
    weight_decay= 0.0005,
    warmup_epochs = 3,
    cos_lr      = True,             # cosine LR scheduler
    augment     = True,
    hsv_h       = 0.015,            # hue augmentation
    hsv_s       = 0.7,
    hsv_v       = 0.4,
    degrees     = 10.0,             # rotation ±10°
    translate   = 0.1,
    scale       = 0.5,
    flipud      = 0.0,
    fliplr      = 0.5,
    mosaic      = 1.0,
    mixup       = 0.1,
    copy_paste  = 0.1,
    rect        = False,
    val         = True,             # run validation each epoch
    plots       = True,             # save training plots
    save        = True,
    save_period = 10,               # save checkpoint every N epochs
    exist_ok    = False,
    verbose     = True,
)

# ──────────────────────────────────────────────
# 3. Evaluate on test set (optional)
# ──────────────────────────────────────────────
best_weights = Path(PROJECT_DIR) / RUN_NAME / "weights" / "best.pt"

print(f"\n[INFO] Loading best weights from: {best_weights}")
best_model = YOLO(str(best_weights))

metrics = best_model.val(
    data   = DATA_YAML,
    split  = "test",               # evaluate on test split
    imgsz  = IMG_SIZE,
    device = DEVICE,
    plots  = True,
)

print("\n── Test-set Metrics ──────────────────────")
print(f"  mAP50      : {metrics.box.map50:.4f}")
print(f"  mAP50-95   : {metrics.box.map:.4f}")
print(f"  Precision  : {metrics.box.mp:.4f}")
print(f"  Recall     : {metrics.box.mr:.4f}")
print("──────────────────────────────────────────\n")

# ──────────────────────────────────────────────
# 4. Export to ONNX
# ──────────────────────────────────────────────
print("[INFO] Exporting model to ONNX …")

onnx_path = best_model.export(
    format      = "onnx",
    imgsz       = IMG_SIZE,
    opset       = 12,              # opset 12 is broadly compatible
    dynamic     = True,            # dynamic batch + spatial dims
    simplify    = True,            # run onnx-simplifier
    half        = False,           # set True for FP16 if GPU supports it
)

print(f"[INFO] ONNX model saved to: {onnx_path}")

# ──────────────────────────────────────────────
# 5. Quick ONNX sanity-check
# ──────────────────────────────────────────────
try:
    import onnx
    model_check = onnx.load(onnx_path)
    onnx.checker.check_model(model_check)
    print("[INFO] ONNX model check PASSED ✓")
except ImportError:
    print("[WARN] 'onnx' package not installed — skipping sanity check.")
    print("       Run:  pip install onnx  to enable it.")
except Exception as e:
    print(f"[ERROR] ONNX check failed: {e}")

print("\n[DONE] Training and export complete.")
print(f"  Best weights : {best_weights}")
print(f"  ONNX model   : {onnx_path}")

[INFO] Using device : 0
[INFO] Data config  : ./data.yaml
[INFO] Epochs       : 100  |  Batch : 16  |  Img size : 640
Ultralytics 8.4.57 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=True, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.1, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=./data.yaml, degrees=10.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.1, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolo

RuntimeError: Dataset 'data.yaml' error ❌ './data.yaml' does not exist